<a href="https://colab.research.google.com/github/GabyPugaBR/agentic-ai-for-developers-concepts-and-applications-for-enterprises-3913172/blob/main/CustomerServiceAgent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install python-dotenv==1.0.0
!pip install llama-index==0.10.59
!pip install llama-index-llms-openai==0.1.27
!pip install llama-index-embeddings-openai==0.1.11
!pip install llama-index --upgrade
!pip install llama-index-llms-openai --upgrade
!pip install llama-index-embeddings-openai --upgrade

  Attempting uninstall: python-dotenv
    Found existing installation: python-dotenv 1.2.1
    Uninstalling python-dotenv-1.2.1:
      Successfully uninstalled python-dotenv-1.2.1
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.3 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of llama-index-indices-managed-llama-cloud to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of llama-index-indices-managed-llama-cloud to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
INFO: pip is looking at multiple versions of llama-index-readers-llama-parse to determine which version is compatible with other requirement

  Attempting uninstall: llama-index-llms-openai
    Found existing installation: llama-index-llms-openai 0.1.31
    Uninstalling llama-index-llms-openai-0.1.31:
      Successfully uninstalled llama-index-llms-openai-0.1.31
^C


In [1]:
from llama_index.llms.openai import OpenAI
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.core import Settings
from google.colab import userdata

from llama_index.core import Settings
import os
import nest_asyncio

nest_asyncio.apply()

Settings.llm = OpenAI(
    model="gpt-4o-mini",  # ← correct model string
    api_key=userdata.get('OpenAI_API')
)

Settings.embed_model = OpenAIEmbedding(
    model="text-embedding-3-small",
    api_key=userdata.get('OpenAI_API')
)

In [3]:
from typing import List
from llama_index.core import SimpleDirectoryReader
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core import  VectorStoreIndex
from llama_index.core.tools import QueryEngineTool

#-------------------------------------------------------------
# Tool 1 : Function that returns the list of items in an order
#-------------------------------------------------------------
def get_order_items(order_id: int) -> List[str] :
    """Given an order Id, this function returns the
    list of items purchased for that order"""

    order_items = {
            1001: ["Laptop","Mouse"],
            1002: ["Keyboard","HDMI Cable"],
            1003: ["Laptop","Keyboard"]
        }
    if order_id in order_items.keys():
        return order_items[order_id]
    else:
        return []

#-------------------------------------------------------------
# Tool 2 : Function that returns the delivery date for an order
#-------------------------------------------------------------
def get_delivery_date(order_id: int) -> str:
    """Given an order Id, this function returns the
    delivery date for that order"""

    delivery_dates = {
            1001: "10-Jun",
            1002: "12-Jun",
            1003: "08-Jun"
    }
    if order_id in delivery_dates.keys():
        return delivery_dates[order_id]
    else:
        return []

#----------------------------------------------------------------
# Tool 3 : Function that returns maximum return days for an item
#----------------------------------------------------------------
def get_item_return_days(item: str) -> int :
    """Given an Item, this function returns the return support
    for that order. The return support is in number of days"""

    item_returns = {
            "Laptop"     : 30,
            "Mouse"      : 15,
            "Keyboard"   : 15,
            "HDMI Cable" : 5
    }
    if item in item_returns.keys():
        return item_returns[item]
    else:
        #Default
        return 45

#-------------------------------------------------------------
# Tool 4 : Vector DB that contains customer support contacts
#-------------------------------------------------------------
#Setup vector index for return policies
support_docs=SimpleDirectoryReader(input_files=["Customer Service.pdf"]).load_data()

splitter=SentenceSplitter(chunk_size=1024)
support_nodes=splitter.get_nodes_from_documents(support_docs)
support_index=VectorStoreIndex(support_nodes)
support_query_engine = support_index.as_query_engine()

In [4]:
from llama_index.core.tools import FunctionTool

#Create tools for the 3 functions and 1 index
order_item_tool = FunctionTool.from_defaults(fn=get_order_items)
delivery_date_tool = FunctionTool.from_defaults(fn=get_delivery_date)
return_policy_tool = FunctionTool.from_defaults(fn=get_item_return_days)

support_tool = QueryEngineTool.from_defaults(
    query_engine=support_query_engine,
    description=(
        "Customer support policies and contact information"
    ),
)

In [5]:
from llama_index.core.agent import FunctionCallingAgentWorker
from llama_index.core.agent import AgentRunner
from llama_index.llms.openai import OpenAI

#Setup the Agent worker in LlamaIndex with all the Tools
#This is the tool executor process
agent_worker = FunctionCallingAgentWorker.from_tools(
    [order_item_tool,
     delivery_date_tool,
     return_policy_tool,
     support_tool
    ],
    llm=Settings.llm,
    verbose=True
)
#Create an Agent Orchestrator with LlamaIndex
agent = AgentRunner(agent_worker)

In [6]:
#Get return policy for an order
response = agent.query(
    "What is the return policy for order number 1001"
)

print("\n Final output : \n", response)

Added user message to memory: What is the return policy for order number 1001
=== Calling Function ===
Calling function: get_order_items with args: {"order_id": 1001}
=== Function Output ===
['Laptop', 'Mouse']
=== Calling Function ===
Calling function: get_item_return_days with args: {"item": "Laptop"}
=== Function Output ===
30
=== Calling Function ===
Calling function: get_item_return_days with args: {"item": "Mouse"}
=== Function Output ===
15
=== LLM Response ===
The return policy for order number 1001 is as follows:

- **Laptop**: You have 30 days to return this item.
- **Mouse**: You have 15 days to return this item.

 Final output : 
 The return policy for order number 1001 is as follows:

- **Laptop**: You have 30 days to return this item.
- **Mouse**: You have 15 days to return this item.


In [7]:
# Three part question
response = agent.query(
    "When is the delivery date and items shipped for order 1003 and how can I contact customer support?"
)

print("\n Final output : \n", response)

Added user message to memory: When is the delivery date and items shipped for order 1003 and how can I contact customer support?
=== Calling Function ===
Calling function: get_order_items with args: {"order_id": 1003}
=== Function Output ===
['Laptop', 'Keyboard']
=== Calling Function ===
Calling function: get_delivery_date with args: {"order_id": 1003}
=== Function Output ===
08-Jun
=== Calling Function ===
Calling function: query_engine_tool with args: {"input": "customer support contact information"}
=== Function Output ===
Customer support is available Monday to Friday from 8 AM to 5 PM. You can contact customer service at 1-987-654-3210 or email support@company.com.
=== LLM Response ===
For order 1003, the items shipped are:

- Laptop
- Keyboard

The delivery date is June 8th.

You can contact customer support Monday to Friday from 8 AM to 5 PM at 1-987-654-3210 or via email at support@company.com.

 Final output : 
 For order 1003, the items shipped are:

- Laptop
- Keyboard

The

In [ ]:
# #Question about an invalid order number
# response = agent.query(
#     "What is the return policy for order number 1004"
# )

# print("\n Final output : \n", response)